# setup

In [ ]:
!pip install -q sb3-contrib


# config.py

In [ ]:
BEGINNER = [9, 9, 10]
INTERMEDIATE = [16, 16, 40]
EXPERT = [16, 30, 99]

LEVEL = INTERMEDIATE


# engine.py

In [ ]:
import numpy as np

MINE = -1   
FLAGGED = -1 
HIDDEN = -2   


# there are 3 boards in total
# one will be the solved with all correct flag and mine count
# one will be the current state of board user can see
# one will be the flag, which will count mine when user place a flag

class MineSweeper:
    def __init__(self, detail, safe_first = False):
        self.rng = np.random.default_rng()
        self.rows = detail[0]
        self.cols = detail[1]
        self.mines = detail[2]
        self.safe_first = safe_first
        self.placed = False
        self.board = np.full((self.rows, self.cols), HIDDEN)
        self.solved_board = self.board.copy()
        self.flag_board = self.board.copy()
        self.exploded = set()
        self.win = False
        self.hidden = 0

    def reset(self):
        self.win = False
        self.hidden = self.rows * self.cols
        self.board = np.full((self.rows, self.cols), HIDDEN)
        self.solved_board = np.zeros((self.rows, self.cols)).astype(int)
        self.flag_board = np.zeros((self.rows, self.cols)).astype(int)
        self.exploded = set()
        self.placed = False

        # safe first click: wait for the first open() to place the mines
        if not self.safe_first:
            self.place_mines()


    def place_mines(self, x = None, y = None):
        # keep the first clicked cell and its neighbours free of mines
        allowed = np.ones((self.rows, self.cols), dtype=bool)
        if x is not None:
            allowed[max(y - 1, 0):y + 2, max(x - 1, 0):x + 2] = False

        temp_board = np.zeros((self.rows, self.cols)).astype(int)
        idx = self.rng.choice(np.flatnonzero(allowed), size=self.mines, replace=False)
        temp_board.flat[idx] = MINE

        mines = (temp_board == MINE).astype(int)
        p = np.pad(mines, 1)


        h, w = mines.shape
        counts = sum(p[dy: dy+h, dx: dx+w]
                    for dy in (0, 1, 2) for dx in (0, 1, 2)
                    if (dx, dy) != (1, 1))

        self.solved_board = np.where(mines == 1, MINE, counts)
        self.placed = True

        
    def flag(self, x, y):
        if self.board[y, x] == HIDDEN:
            self.board[y, x] = FLAGGED

            h, w = self.board.shape
            y0, y1 = max(y - 1, 0), min(y + 2, h)
            x0, x1 = max(x - 1, 0), min(x + 2, w)
            self.flag_board[y0:y1, x0:x1] += 1
            self.flag_board[y, x] -= 1
            return 1

        elif self.board[y, x] == FLAGGED:
            self.board[y, x] = HIDDEN

            h, w = self.board.shape
            y0, y1 = max(y - 1, 0), min(y + 2, h)
            x0, x1 = max(x - 1, 0), min(x + 2, w)
            self.flag_board[y0:y1, x0:x1] -= 1
            self.flag_board[y, x] += 1
            return -1
        
        return 0



    def open(self, x, y, chording = False):
        h, w = self.board.shape
        if x < 0 or y < 0 or x >= w or y >= h:
            return True
        if not self.placed:
            self.place_mines(x, y)
        if self.board[y, x] != HIDDEN:
            if self.board[y, x] == FLAGGED:
                pass
            elif self.board[y, x] != HIDDEN and self.flag_board[y, x] == self.solved_board[y, x] and chording:
                safe = True
                for i in range(max(y - 1, 0), min(y + 2, h)):
                    for j in range(max(x - 1, 0), min(x + 2, w)):
                        if not self.open(j, i):
                            safe = False

                return safe

        elif self.solved_board[y, x] == MINE:
            self.exploded.add((y, x))
            return False
        else:
            stack = [(y, x)]
            while stack:
                cy, cx = stack.pop()
                if self.board[cy, cx] != HIDDEN: continue
                self.board[cy, cx] = self.solved_board[cy, cx]
                self.hidden -= 1
                if self.solved_board[cy, cx] == 0:
                    for i in range(max(cy - 1, 0), min(cy + 2, h)):
                        for j in range(max(cx - 1, 0), min(cx + 2, w)):
                            if (i, j) != (cy, cx):
                                stack.append((i, j))
                                
        # win when the only unopened cells (hidden or flagged) left are the mines
        if self.hidden == self.mines: self.win = True
        return True


# minesweeper_env.py

In [ ]:
import gymnasium as gym
import numpy as np


rows, cols, mines = LEVEL



class MineEnv(gym.Env):
    def __init__(self, safe_first = False):
        self.max_steps = rows * cols
        self.steps = 0
        self.reward_per_open = 1/(rows*cols-mines)
        self.action_space = gym.spaces.Discrete(3*rows*cols)
        self.observation_space = gym.spaces.Box(
            low=0, high = 1,
            shape=(11, rows, cols),
            dtype=np.float32
        )
        self.game = MineSweeper(LEVEL, safe_first)
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)       
        self.game.rng = self.np_random
        self.game.reset()
        self.steps = 0
        return self._obs(), {}
    
    def action_masks(self):
        mask = np.zeros((3, rows, cols), dtype=bool)
        mask[0] = self.game.board == -2
        return mask.reshape(-1)
        
            
    
    def step(self, action):
        hidden_before = self.game.hidden
        info = {}
    
        n = rows*cols
        kind, cell = divmod(int(action), n)
        r, c = divmod(cell, cols)
        
        if kind == 0:
            safe = self.game.open(c, r)
        else:
            safe = True
        opened = hidden_before - self.game.hidden
        if not safe:
            reward, terminated = -1.0, True
        elif self.game.win:
            reward, terminated = 1.0, True
        else:
            reward, terminated = 0.0, False
        
        if opened:         
            # reward += opened*self.reward_per_open
            reward += 0.12
        elif not terminated:
            reward -= 0.1
            
    
        self.steps += 1
        truncated = self.steps >= self.max_steps
        
        if terminated or truncated:
            info["is_success"] = self.game.win
        
        return self._obs(), reward, terminated, truncated, info


    def _obs(self):
        obs = (self.game.board[None, :, :] == np.arange(-2, 9)[:, None, None]).astype(np.float32)
        return obs

# featureCNN.py

In [ ]:
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor



class FeatureCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim = 256):
        super().__init__(observation_space, features_dim)
        n_input = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride = 1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        with torch.no_grad():
            n_flat = self.cnn(torch.zeros(1, *observation_space.shape)).shape[1]
            
            
        self.linear = nn.Sequential(
            nn.Linear(n_flat, features_dim),
            nn.ReLU()
        )
        
    def forward(self, observation):
        output = self.cnn(observation)
        return self.linear(output)


# minePolicy.py

In [ ]:
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy


class ConvTrunk(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim = 64):
        super().__init__(observation_space, features_dim)
        n_input = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride = 1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )


    def forward(self, observation):
        output = self.cnn(observation)
        return output


class ConvHeads(nn.Module):
    def __init__(self, channels, n_kinds, n_actions, vf_dim=64):
        super().__init__()
        self.latent_dim_pi = n_actions
        self.latent_dim_vf = vf_dim

        self.actor = nn.Conv2d(channels, n_kinds, kernel_size=1)

        self.critic = nn.Sequential(
            nn.Linear(channels, vf_dim),
            nn.ReLU(),
        )

    def forward_actor(self, features):
        logits = self.actor(features)
        return logits.flatten(1)

    def forward_critic(self, features):
        pooled = features.mean(dim=(2, 3))
        return self.critic(pooled)

    def forward(self, features):
        return self.forward_actor(features), self.forward_critic(features)


class MinePolicy(MaskableActorCriticPolicy):

    def _build_mlp_extractor(self):
        _, h, w = self.observation_space.shape
        n_actions = self.action_space.n
        n_kinds = n_actions // (h * w)
        self.mlp_extractor = ConvHeads(self.features_dim, n_kinds, n_actions)

    def _build(self, lr_schedule):
        super()._build(lr_schedule)

        self.action_net = nn.Identity()

        if self.ortho_init:
            self.mlp_extractor.actor.apply(lambda m: self.init_weights(m, gain=0.01))

        self.optimizer = self.optimizer_class(self.parameters(), lr=lr_schedule(1), **self.optimizer_kwargs)


# milestone_callback.py

In [ ]:
import os
from stable_baselines3.common.callbacks import BaseCallback


class MilestoneCallback(BaseCallback):
    def __init__(self, milestones, save_path, verbose=0):
        super().__init__(verbose)
        self.milestones = milestones
        self.save_path = save_path     
        self.iteration = 0

    def _on_training_start(self):
        os.makedirs(self.save_path, exist_ok=True)

    def _on_rollout_start(self):
        if self.iteration in self.milestones:
            path = os.path.join(self.save_path, f"iter_{self.iteration}")
            self.model.save(path)
            if self.verbose:
                print(f"Saved milestone {path} at {self.num_timesteps} clicks")
        self.iteration += 1

    def _on_step(self):
        return True


# train.py

In [ ]:
import os
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback
from stable_baselines3.common.callbacks import StopTrainingOnNoModelImprovement, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env


N_ENVS = 4
TOTAL_TIMESTEPS = 15_000_000
RUN = "fullconv_inter_v2"

# /kaggle/working on Kaggle, ./runs on the laptop
BASE_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "runs"
SAVE_DIR = os.path.join(BASE_DIR, RUN)


def make_env():
    env = MineEnv(safe_first=True)
    env = ActionMasker(env, lambda env: env.action_masks())
    return env


if __name__ == "__main__":

    train_env = make_vec_env(make_env, n_envs=N_ENVS, vec_env_cls=SubprocVecEnv)
    eval_env = DummyVecEnv([lambda: Monitor(make_env())])

    # stop = StopTrainingOnNoModelImprovement(
    #     max_no_improvement_evals=10,
    #     min_evals=10,
    #     verbose=1,
    # )

    eval_cb = MaskableEvalCallback(
        eval_env,
        eval_freq=250_000 // N_ENVS,
        n_eval_episodes=300,
        best_model_save_path=os.path.join(SAVE_DIR, "best"),
        log_path=os.path.join(SAVE_DIR, "eval_logs"),
        # callback_after_eval=stop,
        verbose=1,
    )

    checkpoint_cb = CheckpointCallback(
        save_freq=500_000 // N_ENVS,
        save_path=os.path.join(SAVE_DIR, "checkpoints"),
        name_prefix="ckpt",
        verbose=1,
    )

    milestone_cb = MilestoneCallback(
        milestones=[0, 1, 10, 50, 100, 500, 1000, 2000],
        save_path=os.path.join(SAVE_DIR, "milestones"),
        verbose=1,
    )

    policy_kwargs = {
        "features_extractor_class": ConvTrunk,
        "normalize_images": False
    }

    # continue from the intermediate model
    model = MaskablePPO.load("/kaggle/input/fullconv_inter_v1/best/best_model.zip",
                        env=train_env,
                        custom_objects={
                            "observation_space": train_env.observation_space,
                            "action_space": train_env.action_space,
                            "lr_schedule": lambda _: 2e-4,
                            "clip_range": lambda _: 0.2,
                            "policy_class": MinePolicy,
                            "policy_kwargs": policy_kwargs,
                        },
                        learning_rate=2e-4,
                        clip_range=0.2,
                        target_kl=0.02,
                        verbose=1,
                        batch_size=1024,
                        gamma=0.95,
                        ent_coef=0.002)
    print(model.policy)
    model.learn(total_timesteps=TOTAL_TIMESTEPS, 
                callback=[eval_cb, checkpoint_cb, milestone_cb]
                )
    model.save(os.path.join(SAVE_DIR, "final_model"))
